# 🏥 TRICARE RAG Q&A 파이프라인

**사용 데이터**
| 파일 | 형식 | 설명 |
|---|---|---|
| `Costs_Fees.pdf` | PDF | TRICARE 2026 비용·수수료 전체 표 |
| `TOP_Handbook_AUG_2023_FINAL.pdf` | PDF | TRICARE Overseas Program 핸드북 |
| `mental_health_services.csv` | CSV | 정신건강 서비스별 보장 내용 |

**사용 모델 / 도구**
| 구성 요소 | 선택 |
|---|---|
| LLM | `claude-sonnet-4-20250514` |
| 임베딩 | `BAAI/bge-m3` (다국어 100개+) |
| 벡터 DB | Chroma (로컬) |
| PDF 텍스트 | PyMuPDFLoader |
| PDF 표 추출 | pdfplumber |

## 0. 패키지 설치

처음 실행 시에만 필요합니다.

In [1]:
# !pip install langchain langchain-community langchain-chroma langchain-anthropic \
# pymupdf pdfplumber sentence-transformers chromadb python-dotenv -q

## 1. 환경 설정

`.env` 파일 예시:
```
OPENAI_API_KEY=sk-ant-...
```

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')
print(f"✅ API Key 로드: {'성공' if api_key else '실패 → .env 파일 확인 필요'}")

✅ API Key 로드: 성공


## 2. 파일 경로 설정

세 가지 데이터 소스를 함께 사용합니다.
파일이 노트북과 같은 디렉터리에 있으면 파일명만 입력하면 됩니다.

In [ ]:
PDF_COSTS   = 'C:\SKN-_24\제 3차 프로젝트\data\Costs_Fees.pdf'
PDF_HANDBOOK = 'C:\SKN-_24\제 3차 프로젝트\data\TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf'
CSV_MENTAL  = 'C:\SKN-_24\제 3차 프로젝트\data\mental_health_services.csv'

for path in [PDF_COSTS, PDF_HANDBOOK, CSV_MENTAL]:
    status = '✅' if os.path.exists(path) else '❌'
    print(f'{status} {path}')   # 잘 가져왔는지 확인

✅ C:\SKN-_24\제 3차 프로젝트\data\Costs_Fees.pdf
✅ C:\SKN-_24\제 3차 프로젝트\data\TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf
✅ C:\SKN-_24\제 3차 프로젝트\data\mental_health_services.csv


<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:2: SyntaxWarning: invalid escape sequence '\S'
<>:3: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:2: SyntaxWarning: invalid escape sequence '\S'
<>:3: SyntaxWarning: invalid escape sequence '\S'
C:\Users\Playdata\AppData\Local\Temp\ipykernel_12492\3738906588.py:1: SyntaxWarning: invalid escape sequence '\S'
  PDF_COSTS   = 'C:\SKN-_24\제 3차 프로젝트\data\Costs_Fees.pdf'
C:\Users\Playdata\AppData\Local\Temp\ipykernel_12492\3738906588.py:2: SyntaxWarning: invalid escape sequence '\S'
  PDF_HANDBOOK = 'C:\SKN-_24\제 3차 프로젝트\data\TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf'
C:\Users\Playdata\AppData\Local\Temp\ipykernel_12492\3738906588.py:3: SyntaxWarning: invalid escape sequence '\S'
  CSV_MENTAL  = 'C:\SKN-_24\제 3차 프로젝트\data\mental_health_services.csv'


## 3. 임베딩 모델 초기화 — `BAAI/bge-m3`

### BAAI/bge-m3

- Cigna 노트북(은우님 코드) 참고 中 모델 변경
- `BAAI/bge-m3`는 100개 이상의 언어를 **동일한 벡터 공간**에 매핑하므로
한국어 질문과 영어 문서가 높은 유사도로 매칭

In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# GPU가 없는 환경이면 'device': 'cpu'로 변경
embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    #model_kwargs={'device': 'cuda'}  # GPU 환경 -> GPU 없어서 주석 처리
    model_kwargs={'device': 'cpu'}  # CPU 환경
)

print('✅ BAAI/bge-m3 임베딩 모델 로드 완료')

C:\Users\Playdata\AppData\Local\Temp\ipykernel_12492\2704621113.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ BAAI/bge-m3 임베딩 모델 로드 완료


## 4. PDF 로드 — PyMuPDFLoader

### PyMuPDFLoader

- 원본 노트북에서 `PyPDFLoader`는 복잡하게 압축된 최신 PDF에서
- 가끔 파싱 에러가 발생
- `PyMuPDFLoader`는 동일한 페이지 단위 추출 방식이지만
PDF 파싱 안정성이 더 높음

- 두 PDF를 로드한 뒤 하나의 리스트로 합쳐서 청킹
- 메타데이터의 `source` 필드로 어느 파일에서 나온 청크인지 추적할 수 있음

In [5]:
#!pip install pymupdf

In [6]:
from langchain_community.document_loaders import PyMuPDFLoader

loader_costs= PyMuPDFLoader(PDF_COSTS)
loader_handbook= PyMuPDFLoader(PDF_HANDBOOK)

docs_costs= loader_costs.load()
docs_handbook= loader_handbook.load()

# 두 PDF를 하나의 리스트로 합침
all_pdf_docs = docs_costs + docs_handbook

- 확인

In [7]:
print(f'1. Costs_Fees.pdf: {len(docs_costs)}페이지')
print(f'2. TOP_Handbook.pdf: {len(docs_handbook)}페이지')
print(f'=> 합계: {len(all_pdf_docs)}페이지')
print(f'\n--- 샘플 메타데이터 ---')
print(all_pdf_docs[0].metadata)
print(all_pdf_docs[3].metadata)   # Costs_Fees.pdf 마지막 페이지 (총 4페이지)
print(all_pdf_docs[4].metadata)   # TOP_Handbook.pdf 1페이지 (docs_costs 4개 다음부터)

1. Costs_Fees.pdf: 4페이지
2. TOP_Handbook.pdf: 16페이지
=> 합계: 20페이지

--- 샘플 메타데이터 ---
{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2026-01-06T16:05:56-05:00', 'source': 'C:\\SKN-_24\\제 3차 프로젝트\\data\\Costs_Fees.pdf', 'file_path': 'C:\\SKN-_24\\제 3차 프로젝트\\data\\Costs_Fees.pdf', 'total_pages': 4, 'format': 'PDF 1.6', 'title': 'TRICARE 2026 Costs and Fees Sheet', 'author': 'TRICARE Communications', 'subject': '', 'keywords': '', 'moddate': '2026-01-28T17:23:05-05:00', 'trapped': '', 'modDate': "D:20260128172305-05'00'", 'creationDate': "D:20260106160556-05'00'", 'page': 0}
{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2026-01-06T16:05:56-05:00', 'source': 'C:\\SKN-_24\\제 3차 프로젝트\\data\\Costs_Fees.pdf', 'file_path': 'C:\\SKN-_24\\제 3차 프로젝트\\data\\Costs_Fees.pdf', 'total_pages': 4, 'format': 'PDF 1.6', 'title': 'TRICARE 2026 Costs and Fees Sheet', 'author': 'TRICARE Communications', 

## 5. PDF 청킹 — RecursiveCharacterTextSplitter

### chunk_size=500, chunk_overlap=50

- 원본 노트북은 보험 약관 특성상 기본값(200)보다 큰 500을 사용
- TRICARE 문서도 마찬가지로 문장이 길고 맥락이 이어지는 경우 多
- (ex: Group A / Group B 분기 설명이 한 단락에 묶여있는 경우)

- `chunk_overlap=50`은 청크 경계에서 문맥이 끊기는 것을 방지
- ex. "Group A의 본인부담금은—" 뒤에 청크가 잘리면,
- 다음 청크의 앞 50자에 해당 내용이 반복되어 맥락이 유지

1. 분활 설정

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, # 청크 하나의 최대 글자 수(500자 넘으면 자름)
    chunk_overlap=50,   #(앞 청크의 마지막 50자를 다음 청크 앞에서 반복 포함 -> 청크 경계에서 문맥이 끊기는 것 방지)
    separators=['\n\n', '\n', '. ', ' ', '']    #자를 때 우선 순위로 자름. (\n\n: 문단 자르기, \n: 줄바꿈, .: 문장, 그 다음 공백, 마지막 글자 단위)
)

pdf_chunks = text_splitter.split_documents(all_pdf_docs)


- 확인하기

In [ ]:
print(f' 분할 결과: {len(all_pdf_docs)}페이지 → {len(pdf_chunks)}개 청크')
print(f'\n--- 청크 샘플 ---')
print(pdf_chunks[0].page_content[:300])
print(f'\nMetadata: {pdf_chunks[0].metadata}')  #PDF 로더가 알아서 채움

 분할 결과: 20페이지 → 107개 청크

--- 청크 샘플 ---
TRICARE® 2026 Costs and Fees
This is an overview of most TRICARE costs and fees. For detailed costs and fees, including those for 
TRICARE For Life, survivors, and medically retired individuals, visit www.tricare.mil/comparecosts. 
ARE YOU IN GROUP A OR GROUP B? 
• You’re in Group A if your initial 

Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2026-01-06T16:05:56-05:00', 'source': 'C:\\SKN-_24\\제 3차 프로젝트\\data\\Costs_Fees.pdf', 'file_path': 'C:\\SKN-_24\\제 3차 프로젝트\\data\\Costs_Fees.pdf', 'total_pages': 4, 'format': 'PDF 1.6', 'title': 'TRICARE 2026 Costs and Fees Sheet', 'author': 'TRICARE Communications', 'subject': '', 'keywords': '', 'moddate': '2026-01-28T17:23:05-05:00', 'trapped': '', 'modDate': "D:20260128172305-05'00'", 'creationDate': "D:20260106160556-05'00'", 'page': 0}


## 6. CSV 로드 — Row 단위 Document 변환

### CSVLoader 바로 쓰지 않고 직접 변환

- `CSVLoader`를 그냥 쓰면 row 전체를 하나의 Document로 만들어 줌
- `mental_health_services.csv`처럼 `내용` 컬럼이 매우 긴 경우
- 청킹 시 서비스명(Psychotherapy 등)이 잘려나갈 수 있음.

- 직접 변환하면 두 가지를 동시에 보장
1. `page_content`에 `서비스: ...\n내용: ...` 형식으로 항상 서비스명을 포함
2. `metadata`에도 `서비스명`을 저장해서 청킹 후에도 출처 추적 가능

= "Psychotherapy 내용이 500자를 넘어서 청킹되더라도"
분리된 각 청크에 `서비스명: Psychotherapy`가 남아있음


1. csv 읽기

In [10]:
import pandas as pd
from langchain_core.documents import Document

df = pd.read_csv(CSV_MENTAL, encoding='utf-8-sig')

2. Document 변환
- row 하나 -> Document 하나로 변환함
- page_content: 서비스명 직접 포함시킴
    - 각 청크안에 서비스:  Psychotherapy 남게 됨
- metadata: 서비스명 따로 저장 -> 검색 결과 출처 추적 가능함

In [11]:
csv_docs = []
for _, row in df.iterrows():
    doc = Document(
        page_content=f"서비스: {row['서비스']}\n내용: {row['내용']}",   # 임베딩, 검색 대상
        metadata={
            'source': 'mental_health_services.csv',
            '서비스명': row['서비스'],
            'type': 'mental_health'
        }
    )
    csv_docs.append(doc)

3. 긴 row 추가 청킹
- mental_health_services.csv: 컬럼이 row마다 길이 다름
- 세션 종류별 설명이 긴 row: 500자 넘음 -> 잘림
- text_splitter: PDF 청킹할 때 만든거 재사용함

In [12]:
csv_chunks = text_splitter.split_documents(csv_docs)

4. 결과 확인
- 샘플 출력
    - 확인할 포인트: page_content 앞 부분(서비스: ~) 잘 포함됐는지 
    - 확인할 포인트: metadata (서비스: ~) 잘 포함됐는지 

In [13]:
print(f'📋 CSV row 수: {len(df)}개')
print(f'🔪 청킹 후 청크 수: {len(csv_chunks)}개')
print(f'\n--- 청크 샘플 ---')
print(csv_chunks[0].page_content[:300])
print(f'\nMetadata: {csv_chunks[0].metadata}')

📋 CSV row 수: 8개
🔪 청킹 후 청크 수: 12개

--- 청크 샘플 ---
서비스: Psychotherapy

Metadata: {'source': 'mental_health_services.csv', '서비스명': 'Psychotherapy', 'type': 'mental_health'}


In [14]:
# 전체 청크 metadata 순서대로 출력
for i, chunk in enumerate(csv_chunks):
    print(f'[{i}] {chunk.metadata}')

[0] {'source': 'mental_health_services.csv', '서비스명': 'Psychotherapy', 'type': 'mental_health'}
[1] {'source': 'mental_health_services.csv', '서비스명': 'Psychotherapy', 'type': 'mental_health'}
[2] {'source': 'mental_health_services.csv', '서비스명': 'Psychotherapy', 'type': 'mental_health'}
[3] {'source': 'mental_health_services.csv', '서비스명': 'Psychoanalysis', 'type': 'mental_health'}
[4] {'source': 'mental_health_services.csv', '서비스명': 'Psychological testing and assessment', 'type': 'mental_health'}
[5] {'source': 'mental_health_services.csv', '서비스명': 'Psychological testing and assessment', 'type': 'mental_health'}
[6] {'source': 'mental_health_services.csv', '서비스명': 'Psychological testing and assessment', 'type': 'mental_health'}
[7] {'source': 'mental_health_services.csv', '서비스명': 'Psychotropic drugs', 'type': 'mental_health'}
[8] {'source': 'mental_health_services.csv', '서비스명': 'Electroconvulsive therapy', 'type': 'mental_health'}
[9] {'source': 'mental_health_services.csv', '서비스명': 'Coll

## 7. 전체 청크 통합 → Chroma 벡터 저장소

### PDF& CSV -> 하나의 벡터 저장소에 넣기

- 질문 유형: "비용이 얼마야?(PDF)" / "Psychotherapy는 어떻게 보장돼?(CSV) (총 2가지)
- 사용자가 항상 어느 문서에서 답이 나올지 알 수는 없음
- 하나의 저장소로 통합하면 Retriever가 가장 관련 있는 청크를
소스 구분 없이 한 번에 찾게 설정

- 각 청크의 `metadata.source`와 `metadata.type`으로
나중에 어느 파일에서 왔는지 항상 추적할 수 있음

1. PDF 청크 + CSV 청크 합침

In [15]:
from langchain_chroma import Chroma

# PDF 청크 + CSV 청크 합치기
all_chunks = pdf_chunks + csv_chunks

print(f'📦 총 청크 수: {len(all_chunks)}개 (PDF: {len(pdf_chunks)} + CSV: {len(csv_chunks)})')
print('⏳ 벡터 저장소 생성')


📦 총 청크 수: 119개 (PDF: 107 + CSV: 12)
⏳ 벡터 저장소 생성


2. 백터 저장소 생성
- pdf_chunks(PDF 텍스트 청크) + csv_chunk =all_chunks
- collection_name='tricare_rag'

In [16]:
# 백터 저장소 생성
vector_store = Chroma.from_documents(
    documents=all_chunks,   #임배딩할 청크 리스트
    embedding=embedding_model,  #임베딩 모델 / 각 청크 page_content -> 벡터화
    collection_name='tricare_rag',   #Chroma DB에 위 백터들을 묶어두는 이름
    persist_directory='./chrmoa_db'
)

print(f'✅ Chroma 벡터 저장소 생성 완료')
print(f'   저장된 벡터 수: {vector_store._collection.count()}개')

✅ Chroma 벡터 저장소 생성 완료
   저장된 벡터 수: 238개


## 8. Retriever 설정

### k=6 설정

- 원본 노트북(은우님 파일과 동일)에서 k=4에서 k=6으로 올린 이유와 동일
- TRICARE 문서는 **Group A / Group B**, **TRICARE Prime / Select** 등
같은 주제가 여러 청크에 분산되어 있음
- k=4면 한쪽 그룹 정보만 검색될 수 있음
- k=6으로 넉넉하게 설정 ->  LLM이 전체 맥락을 보고 답하게 함
- =>  Group A/B, Prime/Select 등 같은 주제가 여러 청크에 흩어져 있어서 4개보다 6개가 더 안전

In [17]:
# Retriever 설정
retriever = vector_store.as_retriever(
    search_type='similarity',   # 질문 백터와 청크 백터 사이의 코사인 유사도 검색
    search_kwargs={'k': 6}  # 유사도 높은 순서로 상위 6개 청크 가져옴
)

# 동작 확인
test_result = retriever.invoke('TRICARE Prime 본인부담금')  # 질문 -> 검사 단계 (LLM 전달 단계 X)

# 검색 결과 출력
print(f'🔍 테스트 검색: {len(test_result)}개 문서 검색됨')
for i, d in enumerate(test_result):
    source = d.metadata.get('source', 'unknown')    # 파일 출처
    print(f' [{i+1}] {source}: {d.page_content[:80]}...')   # 청크 내용 앞 80자만 출력

🔍 테스트 검색: 6개 문서 검색됨
 [1] C:\SKN-_24\제 3차 프로젝트\data\Costs_Fees.pdf: •	Group B: $462.96 per individual/$927 per family
TRICARE Prime Out-of-Pocket Co...
 [2] C:\SKN-_24\제 3차 프로젝트\data\Costs_Fees.pdf: •	Group B: $462.96 per individual/$927 per family
TRICARE Prime Out-of-Pocket Co...
 [3] C:\SKN-_24\제 3차 프로젝트\data\Costs_Fees.pdf: Group A Retirees and 
their family members
Group B Retirees and their family 
me...
 [4] C:\SKN-_24\제 3차 프로젝트\data\Costs_Fees.pdf: Group A Retirees and 
their family members
Group B Retirees and their family 
me...
 [5] C:\SKN-_24\제 3차 프로젝트\data\Costs_Fees.pdf: monthly or quarterly premium and follow Group B annual 
deductibles and applicab...
 [6] C:\SKN-_24\제 3차 프로젝트\data\Costs_Fees.pdf: monthly or quarterly premium and follow Group B annual 
deductibles and applicab...


## 9. LLM + RAG 체인 구성

### ChatOpenAI = LLM으로 선택

###  temperature=0.1 낮게 설정

- 보험 정보는 오답이 실제 피해로 이어질 수 있기 때문에.
- 해결책: 낮은 temperature는 창의적인 답변 X +문서에 있는 내용을 최대한 그대로 전달하도록 유도

1. llm 초기화

In [18]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model='gpt-4o-mini', 
    temperature=0.1  # 보험 정보 특성상 낮게 설정 → 문서 기반 답변 유도
)

2. 프롬프트 템플릿 정의

In [19]:
from langchain_core.prompts import PromptTemplate

# 역할 및 규칙 지정
PROMPT_TEMPLATE = (
    'You are a TRICARE health benefits specialist.\n'   #페르소나를 설정
    'Answer questions based ONLY on the provided TRICARE documents below.\n'    #학습 데이터에서 알고 있는 내용 외 답변 X
    'If the answer is not in the documents, say "해당 내용은 제공된 문서에서 확인되지 않습니다."\n'
    'Always mention relevant conditions such as Group A/B, plan type, or beneficiary status.\n\n'
    '[참고 문서]\n{context}\n\n'   # ← 검색된 청크가 들어올 자리
    '[질문]\n{question}\n\n'       # ← 사용자 질문이 들어올 자리
    '[답변]\n'
)

rag_prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)

3. format_docs 함수 정의

In [20]:
def format_docs(docs):
    result=[]
    for doc in docs:
        source=doc.metadata.get('source','unkown')
        page=doc.metadata.get('page', '')
        label=f'[{source}{f", p.{page+1}" if page else""}]'
        result.append(f'{label}{doc.page_content}')
    return '\n\n'.join(result) # '\n\n'으로 항목 이어붙이기(join 단독 사용 불가)

4. 입력 분기 정의

In [21]:
from langchain_core.runnables import RunnablePassthrough

input_map = {
    'question':RunnablePassthrough(),       # 질문을 그대로 통과
    'context':retriever | format_docs      # 질문으로 검색 → 포맷
}

5. 파이프(|)로 전체 연결하기

In [22]:
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    input_map        # 질문 → {question, context} dict 생성
    | rag_prompt     # dict → 채워진 프롬프트 문자열
    | model          # 프롬프트 → LLM 응답 객체
    | StrOutputParser()  # 응답 객체 → 순수 텍스트 문자열
)

6. 확인
- 밑에 .invoke()로 불러서 LLM 호출하면 됨

In [23]:
print('✅ RAG 체인 구성 완료 (retriever -> prompt -> gpt-4o-mini →-> parser)')

✅ RAG 체인 구성 완료 (retriever -> prompt -> gpt-4o-mini →-> parser)


## 10. Demo 1 — 일반 LLM vs RAG 비교

**INPUT**
- **질문**: TRICARE Prime에서 정신건강 상담(Psychotherapy)은 어떻게 보장되나요?

**OUPUT**
- **일반 LLM**: 학습 데이터 기반 → 세부 조건(시간 제한, 세션 종류 등) 누락 가능
- **RAG**: `mental_health_services.csv`에서 정확한 내용 검색 후 답변

In [30]:
from langchain_core.messages import HumanMessage

question = (
    'TRICARE Prime에서 정신건강 상담(Psychotherapy)은 어떻게 보장되나요? '
    '세션 종류별(개인/가족/그룹) 시간 제한과 주의사항도 알려주세요.'
)

print('=' * 65)
print('❓ 질문:', question)
print('=' * 65)

print('\n📢 [일반 LLM 답변] — 문서 없이 직접 응답')
print('-' * 65)
plain_response = model.invoke([HumanMessage(content=question)])
print(plain_response.content)
print('\n⚠️  주의: 위 답변은 학습 데이터 기반이며 실제 문서와 다를 수 있습니다.')

❓ 질문: TRICARE Prime에서 정신건강 상담(Psychotherapy)은 어떻게 보장되나요? 세션 종류별(개인/가족/그룹) 시간 제한과 주의사항도 알려주세요.

📢 [일반 LLM 답변] — 문서 없이 직접 응답
-----------------------------------------------------------------
TRICARE Prime에서 정신건강 상담(Psychotherapy)은 다양한 형태로 제공되며, 보장 범위와 세션 종류에 따라 다소 차이가 있습니다. 아래는 TRICARE Prime에서의 정신건강 상담에 대한 일반적인 정보입니다.

### 보장 범위
1. **개인 상담 (Individual Therapy)**: 개인 상담은 일반적으로 가장 많이 이용되는 형태로, 개인의 정신적, 정서적 문제를 다룹니다.
2. **가족 상담 (Family Therapy)**: 가족 구성원 간의 관계 문제나 가족 내에서 발생하는 정신적 문제를 다루는 상담입니다.
3. **그룹 상담 (Group Therapy)**: 여러 사람이 함께 참여하여 서로의 경험을 공유하고 지원하는 형태의 상담입니다.

### 시간 제한
- **개인 상담**: 일반적으로 세션당 50분에서 60분 정도 진행됩니다. 연간 세션 수는 개인의 필요와 치료 계획에 따라 다르지만, TRICARE는 보통 연간 20회까지 보장합니다.
- **가족 상담**: 가족 상담도 비슷한 시간 제한이 있으며, 세션당 50분에서 60분 정도 진행됩니다. 연간 세션 수는 개인 상담과 유사하게 보장됩니다.
- **그룹 상담**: 그룹 상담은 보통 90분에서 120분 정도 진행되며, 그룹의 크기와 주제에 따라 다를 수 있습니다. 연간 세션 수는 치료 계획에 따라 달라질 수 있습니다.

### 주의사항
- **사전 승인**: TRICARE Prime에서는 특정 치료나 세션에 대해 사전 승인이 필요할 수 있습니다. 따라서 상담을 시작하기 전에 반드시 담당 의사나 정신건강 전문가와 상담하여 필요한 절차를 확인해야 

In [31]:
print('\n📚 [RAG 답변] — TRICARE 문서 검색 후 gpt-40-mini 응답')
print('-' * 65)

# 디버그: 어떤 청크가 검색됐는지 확인
retrieved = retriever.invoke(question)
print('🔎 검색된 청크:')
for i, chunk in enumerate(retrieved):
    src = chunk.metadata.get('source', '')
    svc = chunk.metadata.get('서비스명', '')
    print(f'  [{i+1}] {src} {f"({svc})" if svc else ""}: {chunk.page_content[:80]}...')

print()
rag_response = rag_chain.invoke(question)
print(rag_response)

# 리스트 생성 안해서 오류 났었음
#def format_docs(docs):
    #result=[]


📚 [RAG 답변] — TRICARE 문서 검색 후 gpt-40-mini 응답
-----------------------------------------------------------------
🔎 검색된 청크:
  [1] mental_health_services.csv (Psychotherapy): 내용: Psychotherapy is discussion-based mental health therapy. It’s also known as ...
  [2] mental_health_services.csv (Psychotherapy): 내용: Psychotherapy is discussion-based mental health therapy. It’s also known as ...
  [3] mental_health_services.csv (Psychotherapy): . Family or conjoint sessions: Therapy designed to treat the entire family. Regu...
  [4] mental_health_services.csv (Psychotherapy): . Family or conjoint sessions: Therapy designed to treat the entire family. Regu...
  [5] mental_health_services.csv (Psychological testing and assessment): 내용: TRICARE covers psychological testing and assessment to help diagnose a patie...
  [6] mental_health_services.csv (Psychological testing and assessment): 내용: TRICARE covers psychological testing and assessment to help diagnose a patie...

TRICARE Prime에서 정신건강 상담(Psyc

## 11. Demo 2 — 표(Table) 기반 RAG

### 표를 별도로 처리

- `Costs_Fees.pdf`의 비용 테이블은
일반 텍스트 추출 시 열 정렬이 깨짐

- `pdfplumber`는 표의 셀 좌표를 인식
- `행 | 열` 구조를 유지하므로
- ex. "TRICARE Prime, Group A, 외래 진료비 = $26"처럼
정확한 수치 검색이 가능하게 하ㅏㅁ

In [26]:
import pdfplumber

def extract_tables_from_pdf(pdf_path):
    """
    pdfplumber로 PDF 전체에서 표를 추출합니다.
    각 셀은 ' | '로 구분되어 LLM이 행/열 관계를 파악할 수 있습니다.
    """
    table_docs = []
    #pdf 파일 열어서 페이지 순회
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            tables = page.extract_tables()

            for table_idx, table in enumerate(tables):
                if not table or len(table) < 2:  # 헤더+데이터 최소 2행
                    continue
                # 표 셀을 텍스트로 변환
                lines = []
                for row in table:
                    cleaned = [
                        str(cell).strip().replace('\n', ' ') if cell else ''
                        for cell in row
                    ]
                    lines.append(' | '.join(cleaned))

                table_text = '\n'.join(lines)

                #Document 객체 생성
                table_docs.append(Document(
                    page_content=f'[TRICARE 비용표 | p.{page_num + 1}]\n{table_text}',
                    metadata={
                        'source': pdf_path,
                        'page': page_num + 1,
                        'type': 'cost_table',
                        'table_idx': table_idx + 1
                    }
                ))

    return table_docs

# 함수 실행하고 확인
table_docs = extract_tables_from_pdf(PDF_COSTS)
print(f'📊 총 {len(table_docs)}개 표 추출 완료')
print()
print('--- 샘플 ---')
print(table_docs[0].page_content[:400])

📊 총 8개 표 추출 완료

--- 샘플 ---
[TRICARE 비용표 | p.1]
ADSMs, ADFMs, and transitional survivors |  | 
Covered service | Group A | Group B
All covered services | $0 | $0
Retirees, their family members, and all others |  | 
Covered service | Group A | Group B
Preventive care visit | $0 | $0
Primary care outpatient visit | $26 | $26
Specialty care outpatient visit | $39 | $39
Urgent care center visit | $39 | $39
Emergency room visit |


## 12. 표 전용 벡터 저장소 구성

### 표 -> 기존 저장소와 분리

- 비용 관련 수치 질문("TRICARE Prime Group A 외래 진료비는?")
에서는 표 데이터만 검색하는 것이 훨씬 정확
- 텍스트 청크가 섞이면 표와 관련 없는 청크가
k=6 안에 들어와서 노이즈가 될 수 있음

- 용도에 따라 Retriever를 분리하고,
- 복잡한 질문(비용+정책 동시 질문)에서는 두 저장소를 함께 사용

- table_vector_store: PDF에서 pdfplumber로 추출한 표 = table_docs 
- -> tricare_cost_table
- 비용이야기 따로 분리

In [27]:
# 표 전용 백터 저장소 생성
# - 앞에 만든 vector_store(text+csv)와 별도로 표만 따로 저장하는 벡터 저장
table_vector_store = Chroma.from_documents(
    documents=table_docs,
    embedding=embedding_model,
    collection_name='tricare_cost_tables',
    persist_directory='./chroma_db2'
)

# 표 전용 retriever 설정
table_retriever = table_vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}  # 표는 밀도가 높아 3개면 충분
)

# 표 기반 RAG 체인
table_rag_chain = (
    {
        'question': RunnablePassthrough(),
        'context': table_retriever | format_docs
    }
    | rag_prompt
    | model
    | StrOutputParser()
)


print(f'✅ 표 전용 벡터 저장소: {table_vector_store._collection.count()}개 벡터')
print('✅ 표 기반 RAG 체인 구성 완료')

✅ 표 전용 벡터 저장소: 16개 벡터
✅ 표 기반 RAG 체인 구성 완료


In [28]:
# 다음 실행 시 불러오는 방법
#vector_store = Chroma(
#    collection_name='tricare_rag',
#    embedding_function=embedding_model,
#    persist_directory='./chroma_db'
#)

## 13. Demo 2 — 표 기반 비용 Q&A

In [29]:
cost_questions = [
    'TRICARE Prime에서 Group A 재향군인 가족의 응급실 방문 비용은 얼마인가요?',
    'TRICARE Select Group B 재향군인의 네트워크 내 전문의 외래 진료비는 얼마인가요?',
    'TRICARE Reserve Select의 월 보험료(Member only)는 얼마인가요?'
]

for i, q in enumerate(cost_questions, 1):
    print(f'\n{"="*65}')
    print(f'❓ 비용 질문 {i}: {q}')
    print(f'{"="*65}')
    answer = table_rag_chain.invoke(q)
    print(answer)


❓ 비용 질문 1: TRICARE Prime에서 Group A 재향군인 가족의 응급실 방문 비용은 얼마인가요?
Group A 재향군인 가족의 TRICARE Prime에서 응급실 방문 비용은 네트워크 이용 시 $138이며, 비네트워크 이용 시 25%의 비용이 발생합니다.

❓ 비용 질문 2: TRICARE Select Group B 재향군인의 네트워크 내 전문의 외래 진료비는 얼마인가요?
TRICARE Select Group B 재향군인의 네트워크 내 전문의 외래 진료비는 $52입니다.

❓ 비용 질문 3: TRICARE Reserve Select의 월 보험료(Member only)는 얼마인가요?
TRICARE Reserve Select의 월 보험료(Member only)는 $57.88입니다.


## 14. 정리 및 다음 단계

### 데이터 처리 전략 요약

| 데이터 | 처리 방식 | 핵심 이유 |
|---|---|---|
| PDF (텍스트) | PyMuPDFLoader → 청킹 | 안정적 파싱, 페이지 메타데이터 자동 포함 |
| PDF (표) | pdfplumber → Document | 행/열 구조 보존, 수치 정확도 보장 |
| CSV | 직접 row 변환 → 청킹 | 서비스명을 본문+메타데이터 양쪽에 보존 |

### Demo 결과 요약

| | 일반 LLM | RAG (텍스트) | RAG (표) |
|---|---|---|---|
| 데이터 출처 | 학습 데이터 | TRICARE PDF + CSV | Costs_Fees.pdf 표 |
| 정확도 | 낮음 | 높음 | 매우 높음 (수치) |
| 한국어 질문 | 가능 | ✅ bge-m3로 지원 | ✅ bge-m3로 지원 |

### 향후 개선 방향

1. **메타데이터 필터링** — Group A/B, 플랜 종류 필터를 Retriever에 추가
2. **앙상블 Retriever** — BM25(키워드) + bge-m3(의미) 혼합 검색
3. **Persistent Chroma** — `persist_directory` 설정으로 재실행 시 재임베딩 생략
4. **평가 파이프라인** — RAGAS 라이브러리로 답변 품질 정량 측정